# SePar: Sequence Pathfinder for Multi-Agent Pickup and Delivery (MAPD)
## Research Reproduction & Demonstration Notebook

This notebook demonstrates the end-to-end implementation of the **SePar (Sequence Pathfinder)** framework:
1. **Perception Backbone**: Spatial-temporal feature extractor using VGG convolutional blocks + LSTM recurrent memory (`src/models.py`).
2. **Multi-Agent Transformer (MAT)**: Autoregressive transformer encoder-decoder with causal masking and discrete action embeddings (`src/models.py`).
3. **Hybrid Policy Optimization**: On-policy Proximal Policy Optimization (PPO) with Generalized Advantage Estimation (GAE) and Imitation Learning (IL) (`src/trainer.py`).
4. **Empirical Evaluation**: Realistic throughput, success rate, and latency benchmarking in multi-agent grid environments (`src/evaluate.py`).

In [ ]:
import logging
import torch
import matplotlib.pyplot as plt

from src.config import Config
from src.models import PolicyNetwork
from src.environment import create_env
from src.trainer import RolloutBuffer, SeParTrainer
from src.evaluate import evaluate_policy, plot_training_history

logging.basicConfig(level=logging.INFO, format="%(asctime)s [%(levelname)s] %(message)s")
config = Config(num_agents=8, train_iterations=3, rollout_steps=64)
print(f"Hardware device: {config.device}")
print(f"Simulation environment: {config.env_name} | Number of agents: {config.num_agents}")

In [ ]:
# Initialize unified policy network
policy = PolicyNetwork(config).to(config.device)
total_params = sum(p.numel() for p in policy.parameters() if p.requires_grad)
print("PolicyNetwork initialized successfully.")
print(f"Total trainable parameters: {total_params:,}")

# Verify forward pass tensor dimensions
dummy_obs = torch.randn(config.num_agents, 3, config.fov_size, config.fov_size, device=config.device)
h = torch.zeros(config.num_agents, config.lstm_hidden, device=config.device)
c = torch.zeros(config.num_agents, config.lstm_hidden, device=config.device)

actions, log_probs, values, (h_next, c_next) = policy.act(dummy_obs, h, c)
print(f"Autoregressive action sampling: {actions.shape}, Values: {values.shape}")

In [ ]:
# Initialize environment and execute sample step
env = create_env(config=config)
obs = env.reset()
print(f"Observation tensor shape: {obs.shape}")

next_obs, rewards, dones, infos = env.step(actions)
print(f"Step complete. Rewards sum: {rewards.sum().item():.3f} | Infos: {infos}")

In [ ]:
# Setup rollout buffer and trainer for on-policy PPO
buffer = RolloutBuffer(
    buffer_size=config.rollout_steps,
    num_agents=config.num_agents,
    obs_shape=(3, config.fov_size, config.fov_size),
    hidden_dim=config.hidden_dim,
    device=config.device
)
trainer = SeParTrainer(policy, config)

print(f"Starting training loop: {config.train_iterations} iterations...")
history = {"policy_loss": [], "value_loss": [], "entropy": [], "rl_loss": [], "returns": []}

for iteration in range(1, config.train_iterations + 1):
    buffer.reset()
    obs = env.reset()
    h = torch.zeros(config.num_agents, config.lstm_hidden, device=config.device)
    c = torch.zeros(config.num_agents, config.lstm_hidden, device=config.device)
    ep_return = 0.0

    for step in range(config.rollout_steps):
        with torch.no_grad():
            actions, log_probs, values, (h_next, c_next) = policy.act(obs, h, c)
        next_obs, rewards, dones, _ = env.step(actions)
        buffer.add(obs, actions, rewards, dones, values, log_probs, h, c)
        ep_return += rewards.sum().item()
        obs, h, c = next_obs, h_next, c_next

    with torch.no_grad():
        _, last_values, _ = policy(obs, h, c)
        buffer.compute_returns_and_advantages(last_values, torch.zeros(config.num_agents, device=config.device))

    metrics = trainer.train_step(buffer)
    for k in ["policy_loss", "value_loss", "entropy"]:
        history[k].append(metrics[k])
    history["rl_loss"].append(metrics["total_rl_loss"])
    history["returns"].append(ep_return)
    print(f"Iter {iteration}/{config.train_iterations} | Policy Loss: {metrics['policy_loss']:.4f} | Value Loss: {metrics['value_loss']:.4f} | Return: {ep_return:.2f}")

In [ ]:
# Empirical evaluation and metric plotting
print("Running empirical evaluation across episodes...")
eval_results = evaluate_policy(env, policy, num_episodes=config.eval_episodes, device=config.device)
print("\n--- Evaluation Results ---")
for k, v in eval_results.items():
    print(f"{k}: {v}")

plot_training_history(history, save_dir="benchmarks")
print("Training history plot saved to benchmarks/training_progress.png")